# Restricted-input method reference
Source: gps_analysis/general_model/duration_analysis/main.ipynb
Requires independently obtained inputs and source directory layout. Not an end-to-end pipeline. Outputs can contain sensitive data; keep the working directory private. External absolute paths have been replaced by external_inputs/<basename>. See docs/REPRODUCIBILITY.md.


In [ ]:
import os
from pathlib import Path
private_root = Path(os.environ["UFUG_PRIVATE_ROOT"]).resolve()
os.chdir(private_root)


In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import os

In [ ]:
## SET THE WORKING DIRECTORY
# Working directory is explicitly configured in the first cell.

In [ ]:
for park in ['amk', 'jurong', 'punggol']:
    print(f"Processing {park} park...")
    
    park_hulls = gpd.read_file(f'gis_files/hulls/{park}/breaks_hulls.geojson')

    park_hulls_centroid = park_hulls.to_crs(epsg=3414)

    park_hulls_centroid.drop(columns=['path'], inplace=True)
    # use centroid of hulls to spatially join with segmented hexagons for further analysis
    park_hulls_centroid['geometry'] = park_hulls_centroid.geometry.centroid

    del park_hulls

    segmented_hex = gpd.read_file(f"gis_files/hexagons/segmented/{park}_hex_agg.geojson")

    segmented_hex = segmented_hex[['grid_id', 'geometry']]

    # segmented_hex.drop(columns=['index_right'], inplace=True)

    # SPATIAL JOIN
    segmented_hex_joined = gpd.sjoin(segmented_hex, park_hulls_centroid, how='left', predicate='intersects')

    segmented_hex_joined['start_time'] = pd.to_datetime(segmented_hex_joined['start_time'], errors='coerce')
    segmented_hex_joined['end_time'] = pd.to_datetime(segmented_hex_joined['end_time'], errors='coerce')

    df_med = segmented_hex_joined.dropna(subset=['start_time'])
    print(df_med['start_time'].dt.date.min(), "to", df_med['start_time'].dt.date.max())

    F_h = segmented_hex_joined.groupby('grid_id')['cluster'].nunique()
    merged = pd.merge(segmented_hex, F_h.rename('F_h'), on='grid_id')
    duration_sum = segmented_hex_joined.groupby('grid_id')['duration'].sum()
    merged = pd.merge(merged, duration_sum.rename('D_h'), on='grid_id')
    merged.to_file(f'gps_analysis/general_model/duration_analysis/{park}_duration_analysis.geojson', driver='GeoJSON')